In [ ]:
OPENAI_API_KEY = "key_here"

In [1]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain.agents import create_agent
import math

# Step1: Without memory

In [5]:
# 1) 
# LLM
llm = ChatOpenAI(
    model="gpt-4o-mini",
    api_key=OPENAI_API_KEY,
    temperature=0
)

# -------------------------
# TOOL
# -------------------------
@tool
def calculate_emi(principal: float, rate: float, time: float) -> dict:
    """Calculate EMI for a loan."""
    print("...Running EMI tool")

    R = rate / 12 / 100
    N = time * 12

    emi = (principal * R * (1 + R)**N) / ((1 + R)**N - 1)

    return {
        "principal": principal,
        "rate": rate,
        "time": time,
        "emi": round(emi, 2)
    }

agent = create_agent(
    model=llm,
    tools=[calculate_emi]
)


In [9]:
# 2) First query
response = agent.invoke({
    "messages": [
        {"role": "user", 
         "content": "I took a loan of 2 lakh at 10% for 2 years. What is my EMI?"}
    ]
})


print("Response:", response)
print("Final Answer:", response["messages"][-1].content)
print("---------------------")
for msg in response["messages"]:
    if msg.type == "tool":
        print("Tool Output:", msg.content)


...Running EMI tool
Response: {'messages': [HumanMessage(content='I took a loan of 2 lakh at 10% for 2 years. What is my EMI?', additional_kwargs={}, response_metadata={}, id='2b13be21-42e4-4ccf-9fd6-0610e472e084'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 24, 'prompt_tokens': 74, 'total_tokens': 98, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_de7acce317', 'id': 'chatcmpl-DZtsrDwlAmEVBCBvtgxr2kgKgCZ67', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019dd832-4bed-7211-8584-153a31202a4a-0', tool_calls=[{'name': 'calculate_emi', 'args': {'principal': 200000, 'rate': 10, 'time': 2}, 'id': 'call_gbClHWiaar7R3Yi09iQC5Hgq', 't

In [10]:
# 3) Follow-up query fails

response = agent.invoke({
    "messages": [
        {"role": "user", 
         "content": "Now if interest changes to 20%, then what is my EMI?"}
    ]
})


print("Response:", response)
print("Final Answer:", response["messages"][-1].content)
print("---------------------")
for msg in response["messages"]: # May hallucinate
    if msg.type == "tool":
        print("Tool Output:", msg.content)


...Running EMI tool
Response: {'messages': [HumanMessage(content='Now if interest changes to 20%, then what is my EMI?', additional_kwargs={}, response_metadata={}, id='18ca6277-8184-43b6-94c9-689b17c49f9a'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 24, 'prompt_tokens': 66, 'total_tokens': 90, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_de7acce317', 'id': 'chatcmpl-DZtt8yZ8sZaapj4ui8wXj0z3ilSEN', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019dd832-8d41-7ae2-b581-929a019640ca-0', tool_calls=[{'name': 'calculate_emi', 'args': {'principal': 100000, 'rate': 20, 'time': 5}, 'id': 'call_cAPHsQBcenm7WJsm5m4pY5OS', 'type': '

# Problem: 
- Missing context → model fills gaps with plausible defaults